In [ ]:
# --- ΒΗΜΑ 0: ΕΓΚΑΤΑΣΤΑΣΗ ΒΙΒΛΙΟΘΗΚΩΝ ---
print("⏳ Εγκατάσταση απαραίτητων βιβλιοθηκών...")
!pip install torch_geometric torch scikit-learn pandas numpy
print("✅ Η εγκατάσταση ολοκληρώθηκε.")

⏳ Εγκατάσταση απαραίτητων βιβλιοθηκών...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.5 MB/s eta 0:00:00
✅ Η εγκατάσταση ολοκληρώθηκε.


In [ ]:
# --- ΒΗΜΑ 1: ΕΓΚΑΤΑΣΤΑΣΗ TRANSFORMERS ---
print("⏳ Εγκατάσταση Hugging Face Transformers...")
!pip install transformers
print("✅ Έτοιμοι να κατεβάσουμε το ProtBERT!")

⏳ Εγκατάσταση Hugging Face Transformers...
✅ Έτοιμοι να κατεβάσουμε το ProtBERT!


In [ ]:
# --- ΒΗΜΑ 2: ΦΟΡΤΩΣΗ PROT-BERT ---
from transformers import BertModel, BertTokenizer
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm # Για να βλέπουμε μπάρα προόδου

# Έλεγχος για GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Χρησιμοποιούμε: {device}")

print("⏳ Φόρτωση του μοντέλου ProtBERT (μπορεί να πάρει 1-2 λεπτά)...")

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
bert_model = BertModel.from_pretrained("Rostlab/prot_bert").to(device)
bert_model.eval() # Το βάζουμε σε mode αξιολόγησης (όχι εκπαίδευσης)

print("✅ Το ProtBERT φορτώθηκε επιτυχώς!")

⚙️ Χρησιμοποιούμε: cuda
⏳ Φόρτωση του μοντέλου ProtBERT (μπορεί να πάρει 1-2 λεπτά)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

✅ Το ProtBERT φορτώθηκε επιτυχώς!


In [ ]:
# --- ΒΗΜΑ ΑΝΑΚΤΗΣΗΣ ΔΕΔΟΜΕΝΩΝ ---
import pandas as pd

print("⏳ Επαναφόρτωση δεδομένων...")

# 1. Φόρτωση αρχείων (Βεβαιώσου ότι τα csv είναι ανεβασμένα αριστερά)
try:
    df_pos = pd.read_csv('positive_protein_sequences.csv')
    df_neg = pd.read_csv('negative_protein_sequences.csv')

    # 2. Ετικέτες
    df_pos['Label'] = 1
    df_neg['Label'] = 0

    # 3. Ένωση και Ανακάτεμα
    df = pd.concat([df_pos, df_neg], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"✅ Τα δεδομένα φορτώθηκαν ξανά! ({len(df)} ζεύγη)")

except FileNotFoundError:
    print("❌ ΣΦΑΛΜΑ: Δεν βρέθηκαν τα αρχεία csv. Ανέβασέ τα ξανά στο Colab!")

⏳ Επαναφόρτωση δεδομένων...
✅ Τα δεδομένα φορτώθηκαν ξανά! (73110 ζεύγη)


In [ ]:
# --- ΒΗΜΑ 3 (ΔΙΟΡΘΩΜΕΝΟ): ΑΣΦΑΛΗΣ ΕΞΑΓΩΓΗ EMBEDDINGS ---
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# 1. Καθαρισμός Μνήμης GPU (για να μην βγάλει Out Of Memory)
torch.cuda.empty_cache()

# 2. Συνάρτηση (Πιο ανθεκτική)
def get_protbert_embeddings_safe(sequences, batch_size=4): # ΜΕΙΩΣΑΜΕ ΤΟ BATCH SIZE ΣΕ 4
    embeddings = []

    # Μετατροπή όλων σε string για σιγουριά
    sequences = [str(s) for s in sequences]

    for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting..."):
        try:
            batch_seqs = sequences[i : i+batch_size]

            # Βάζουμε κενά ανάμεσα στα αμινοξέα
            batch_seqs_spaced = [" ".join(list(seq)) for seq in batch_seqs]

            # Tokenization (Κόβουμε ό,τι είναι πάνω από 512 χαρακτήρες για να μην σκάσει)
            inputs = tokenizer(batch_seqs_spaced, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = bert_model(**inputs)
                # Παίρνουμε τον μέσο όρο (Mean Pooling)
                seq_vectors = outputs.last_hidden_state.mean(dim=1)

            embeddings.append(seq_vectors.cpu().numpy())

            # Καθαρίζουμε λίγη μνήμη σε κάθε βήμα
            del inputs, outputs, seq_vectors
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"⚠️ Σφάλμα στο batch {i}: {str(e)}")
            # Αν σκάσει, βάλε μηδενικά για να μην σταματήσει όλο το πρόγραμμα
            embeddings.append(np.zeros((len(batch_seqs), 1024)))

    return np.vstack(embeddings)

# 3. Προετοιμασία Μικρού Δείγματος (1000 ζεύγη)
# Αφαιρούμε τυχόν προβληματικές γραμμές
df_small = df.sample(1000, random_state=42).dropna().reset_index(drop=True)

print(f"🧬 Ξεκινάμε την ανάλυση για {len(df_small)} ζεύγη...")

# Εξαγωγή για την Πρωτεΐνη 1
print("1️⃣ Επεξεργασία Πρωτεΐνης Α...")
emb_1 = get_protbert_embeddings_safe(df_small['protein_sequences_1'].tolist())

# Εξαγωγή για την Πρωτεΐνη 2
print("2️⃣ Επεξεργασία Πρωτεΐνης Β...")
emb_2 = get_protbert_embeddings_safe(df_small['protein_sequences_2'].tolist())

# Ενώνουμε
if len(emb_1) == len(emb_2):
    X_bert = np.hstack((emb_1, emb_2))
    y_bert = df_small['Label'].values
    print(f"\n✅ ΕΤΟΙΜΟ! Πίνακας χαρακτηριστικών: {X_bert.shape}")
else:
    print("❌ Κάτι πήγε στραβά με τα μεγέθη των πινάκων.")

🧬 Ξεκινάμε την ανάλυση για 1000 ζεύγη...
1️⃣ Επεξεργασία Πρωτεΐνης Α...


Extracting...: 100%|██████████| 250/250 [02:25<00:00,  1.72it/s]


2️⃣ Επεξεργασία Πρωτεΐνης Β...


Extracting...: 100%|██████████| 250/250 [02:21<00:00,  1.77it/s]


✅ ΕΤΟΙΜΟ! Πίνακας χαρακτηριστικών: (1000, 2048)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- ΒΗΜΑ 4: ΕΚΠΑΙΔΕΥΣΗ HYBRID MODEL (ProtBERT + MLP) ---
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

print("🚀 Εκπαίδευση του Υβριδικού Μοντέλου...")

# 1. Χωρίζουμε σε Train (80%) και Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X_bert, y_bert, test_size=0.2, random_state=42)

# 2. Ορίζουμε τον Ταξινομητή (Ένα δυνατό Νευρωνικό Δίκτυο)
# hidden_layer_sizes=(128, 64): Δύο επίπεδα νευρώνων για να μάθει τις σχέσεις
clf = MLPClassifier(hidden_layer_sizes=(128, 64),
                    activation='relu',
                    solver='adam',
                    max_iter=500,
                    random_state=42)

# 3. Εκπαίδευση
clf.fit(X_train, y_train)

# 4. Πρόβλεψη
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1] # Πιθανότητες για το ROC

# 5. Αποτελέσματα
acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)

print("\n🏆 ΑΠΟΤΕΛΕΣΜΑΤΑ (ProtBERT + MLP):")
print("---------------------------------------")
print(f"✅ Accuracy:  {acc*100:.2f}%")
print(f"✅ AUC Score: {auc:.4f}")
print("---------------------------------------")
print("\nΛεπτομερής Αναφορά:")
print(classification_report(y_test, preds))

🚀 Εκπαίδευση του Υβριδικού Μοντέλου...

🏆 ΑΠΟΤΕΛΕΣΜΑΤΑ (ProtBERT + MLP):
---------------------------------------
✅ Accuracy:  79.50%
✅ AUC Score: 0.8978
---------------------------------------

Λεπτομερής Αναφορά:
              precision    recall  f1-score   support

           0       0.76      0.82      0.79        94
           1       0.83      0.77      0.80       106

    accuracy                           0.80       200
   macro avg       0.80      0.80      0.79       200
weighted avg       0.80      0.80      0.80       200



# part 2    ESM2

In [ ]:
# --- ΒΗΜΑ 1: ΣΥΝΔΕΣΗ ΜΕ GOOGLE DRIVE ---
from google.colab import drive
drive.mount('/content/drive')

# Δημιουργία φακέλου για να τα βάζουμε όλα μέσα
import os
save_path = '/content/drive/My Drive/Diplomatiki_ESM2'
os.makedirs(save_path, exist_ok=True)
print(f"✅ Ο φάκελος δημιουργήθηκε: {save_path}")

Mounted at /content/drive
✅ Ο φάκελος δημιουργήθηκε: /content/drive/My Drive/Diplomatiki_ESM2


In [ ]:
# --- ΒΗΜΑ 2: ΦΟΡΤΩΣΗ ΠΛΗΡΟΥΣ DATASET ---
import pandas as pd

# Φόρτωση
try:
    df_pos = pd.read_csv('positive_protein_sequences.csv')
    df_neg = pd.read_csv('negative_protein_sequences.csv')

    # Ετικέτες (1 = Αλληλεπίδραση, 0 = Όχι)
    df_pos['Label'] = 1
    df_neg['Label'] = 0

    # Ένωση
    df_full = pd.concat([df_pos, df_neg], ignore_index=True)

    # Ανακάτεμα (Shuffle) για να μην είναι όλα τα θετικά στην αρχή
    df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"✅ Το πλήρες dataset φορτώθηκε!")
    print(f"📊 Σύνολο ζευγών: {len(df_full)}")
    print(f"   - Θετικά: {len(df_full[df_full['Label']==1])}")
    print(f"   - Αρνητικά: {len(df_full[df_full['Label']==0])}")

except Exception as e:
    print(f"❌ Λάθος: {e}")
    print("Βεβαιώσου ότι ανέβασες τα csv αρχεία στα Files αριστερά!")

✅ Το πλήρες dataset φορτώθηκε!
📊 Σύνολο ζευγών: 73110
   - Θετικά: 36630
   - Αρνητικά: 36480


In [ ]:
# --- ΒΗΜΑ 3: ΕΓΚΑΤΑΣΤΑΣΗ & ΦΟΡΤΩΣΗ ESM-2 ---
# Εγκατάσταση βιβλιοθήκης
!pip install transformers

from transformers import AutoTokenizer, EsmModel
import torch
import numpy as np
from tqdm import tqdm

# Ρύθμιση GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Χρησιμοποιούμε: {device}")

# Φόρτωση του ESM-2 (35M parameters - ιδανικό για Colab)
model_name = "facebook/esm2_t12_35M_UR50D"

print("⏳ Κατέβασμα μοντέλου ESM-2 (Facebook)...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)
model.eval() # Mode αξιολόγησης

print("✅ Το μοντέλο ESM-2 είναι έτοιμο!")

⚙️ Χρησιμοποιούμε: cuda
⏳ Κατέβασμα μοντέλου ESM-2 (Facebook)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Το μοντέλο ESM-2 είναι έτοιμο!


In [ ]:
# --- ΒΗΜΑ 4: ΜΑΖΙΚΗ ΕΞΑΓΩΓΗ EMBEDDINGS (BATCH PROCESSING) ---

def get_esm2_embeddings(sequences, batch_size=8):
    embeddings = []

    # Καθαρισμός δεδομένων (μετατροπή σε string)
    sequences = [str(seq) for seq in sequences]

    for i in tqdm(range(0, len(sequences), batch_size), desc="Processing ESM-2"):
        batch = sequences[i : i + batch_size]

        try:
            # Tokenization
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                # Παίρνουμε το [CLS] token (το πρώτο token που περιέχει την πληροφορία όλης της πρωτεΐνης)
                # Στο ESM συνήθως παίρνουμε το mean (μέσο όρο) ή το CLS. Εδώ θα πάρουμε το Mean.
                embeddings_batch = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

            embeddings.append(embeddings_batch)

        except Exception as e:
            print(f"⚠️ Error in batch {i}: {e}")
            # Αν σκάσει, βάλε μηδενικά
            embeddings.append(np.zeros((len(batch), 480))) # 480 είναι η διάσταση του ESM-2-35M

    return np.vstack(embeddings)

print("🚀 Ξεκινάμε την εξαγωγή για την Πρωτεΐνη 1...")
emb_1 = get_esm2_embeddings(df_full['protein_sequences_1'].tolist())

# Αποθήκευση στο Drive (Ενδιάμεσα)
np.save(f'{save_path}/esm2_embeddings_1.npy', emb_1)
print("✅ Τα embeddings της Πρωτεΐνης 1 σώθηκαν στο Drive!")

print("\n🚀 Ξεκινάμε την εξαγωγή για την Πρωτεΐνη 2...")
emb_2 = get_esm2_embeddings(df_full['protein_sequences_2'].tolist())

# Αποθήκευση στο Drive (Ενδιάμεσα)
np.save(f'{save_path}/esm2_embeddings_2.npy', emb_2)
print("✅ Τα embeddings της Πρωτεΐνης 2 σώθηκαν στο Drive!")

# Αποθήκευση και των Labels για σιγουριά
np.save(f'{save_path}/labels.npy', df_full['Label'].values)

print("\n🎉 ΤΕΛΟΣ! Όλα τα δεδομένα έχουν μετατραπεί και αποθηκευτεί στο Drive.")

🚀 Ξεκινάμε την εξαγωγή για την Πρωτεΐνη 1...


Processing ESM-2: 100%|██████████| 9139/9139 [23:58<00:00,  6.35it/s]


✅ Τα embeddings της Πρωτεΐνης 1 σώθηκαν στο Drive!

🚀 Ξεκινάμε την εξαγωγή για την Πρωτεΐνη 2...


Processing ESM-2: 100%|██████████| 9139/9139 [23:19<00:00,  6.53it/s]


✅ Τα embeddings της Πρωτεΐνης 2 σώθηκαν στο Drive!

🎉 ΤΕΛΟΣ! Όλα τα δεδομένα έχουν μετατραπεί και αποθηκευτεί στο Drive.
